# Part 2 — RAG narrative layer walkthrough

Shows the knowledge base, one narrated case (grounded, cited), and the
verification summary — by loading the artifacts `rag_narrate.py` produces and the
tested code in `src/oct_cds/rag/`. No reimplemented logic.

Generate the artifacts first (from the repo root):

```bash
pip install -e ".[rag]"
python rag_narrate.py paths=kaggle rag_run.split=external_test
```

Set `OCT_CDS_ENV` (`kaggle` or `default`). See [PART2.md](../PART2.md).

In [ ]:
import json, os, sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

ENV = os.environ.get("OCT_CDS_ENV", "kaggle")
SPLIT = "external_test"

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

GlobalHydra.instance().clear()
with initialize_config_dir(version_base=None, config_dir=str(REPO / "configs")):
    cfg = OmegaConf.to_container(
        compose(config_name="config", overrides=[f"paths={ENV}", f"paths.root_dir={REPO.as_posix()}"]),
        resolve=True,
    )
GlobalHydra.instance().clear()

RAG_DIR = Path(cfg["output_dir"]) / "rag"
print("env      :", ENV)
print("rag dir  :", RAG_DIR)


def load_json(p):
    p = Path(p)
    if not p.exists():
        print(f"  (missing: {p})")
        return None
    return json.loads(p.read_text())


def show(df):
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df)

## 1 · Knowledge base

6 entries, NEI (public domain) + one CC-BY review. AAO Preferred Practice
Patterns and EyeWiki are deliberately excluded (their terms forbid use in an AI
system). Each `##` section is one retrievable, citable passage.

In [ ]:
from oct_cds.rag.ingest import load_knowledge_base

kb = load_knowledge_base()
rows = []
for eid in sorted({p.entry_id for p in kb.passages}):
    ps = [p for p in kb.passages if p.entry_id == eid]
    rows.append({
        "entry": f"{eid}.md",
        "title": ps[0].entry_title,
        "covers": ", ".join(ps[0].covers),
        "passages": len(ps),
        "has_model_behavior_note": any(p.is_model_behavior_note for p in ps),
        "sources": " | ".join(ps[0].sources),
    })
show(pd.DataFrame(rows))
print(f"\n{len(kb.passages)} passages total; covers_map = {kb.covers_map}")
print("passage ids:", [p.id for p in kb.passages])

## 2 · A narrated case

One case from `narratives_<split>.jsonl`. The **impression** and **triage** are
verbatim from Part 1's rule engine; `narrative_rag` is the model's grounded
explanation; every bracketed id resolves to a retrieved passage.

In [ ]:
SAMPLE_STEM = None   # None -> first verified case; or set e.g. "CNV__p0"

jf = RAG_DIR / f"narratives_{SPLIT}.jsonl"
cases = [json.loads(l) for l in jf.read_text().splitlines() if l.strip()] if jf.exists() else []
if not cases:
    print(f"run:  python rag_narrate.py paths={ENV} rag_run.split={SPLIT}")
else:
    verified = [c for c in cases if c.get("narrator_meta", {}).get("verified")]
    pick = None
    if SAMPLE_STEM:
        pick = next((c for c in cases if Path(c["case"]["image_path"]).stem == SAMPLE_STEM), None)
    pick = pick or (verified[0] if verified else cases[0])

    m = pick["narrator_meta"]
    print("IMAGE     :", pick["case"]["image_path"], " true:", pick.get("true_class"))
    print("IMPRESSION: {predicted_class}  (confidence {confidence:.0%})  [from Part 1 rule engine]".format(**pick["impression"]))
    print("DIFFERENTIAL:", ", ".join(f"{d['class']} {d['probability']:.0%}" for d in pick["differential"]))
    print("TRIAGE    : {urgency}  — {recommendation}  [from Part 1 rule engine]".format(**pick["triage"]))
    print(f"VERIFIED  : {m['verified']}   flags: {m['flags']}   model: {m['model']}")
    print("\n── narrative_rag ──\n")
    print(pick.get("narrative_rag", "(fell back to Part 1 template)"))
    print("\n── citations ──")
    for c in pick.get("citations", []):
        print(f"  [{c['id']}]  {c['label']}")
        for s in c["sources"]:
            print(f"       - {s}")
    print("\nretrieved passage ids:", m["retrieved_ids"])

## 3 · Verification summary

From `summary_<split>.json`. Every eligible case must produce a narrative that
passes `verify.py`, or it falls back to Part 1's template — nothing unverified is
ever shown.

In [ ]:
s = load_json(RAG_DIR / f"summary_{SPLIT}.json")
if s:
    oc = s["outcomes"]
    show(pd.DataFrame(sorted(oc.items()), columns=["outcome", "n"]))
    eligible = sum(v for k, v in oc.items() if not k.startswith("skipped"))
    verified = oc.get("verified", 0) + oc.get("verified_with_flags", 0)
    fell_back = oc.get("fell_back", 0)
    print(f"\nbackend: {s['backend']}")
    print(f"eligible for narration: {eligible}")
    print(f"verified: {verified}/{eligible}"
          + (f"  ({verified/eligible:.0%})" if eligible else ""))
    print(f"fell back to Part 1 template: {fell_back}")
    print(f"flag counts: {s.get('flag_counts', {})}")
else:
    print(f"run:  python rag_narrate.py paths={ENV} rag_run.split={SPLIT}")

---

The language model explains; it never decides. The impression and triage in
every report come verbatim from Part 1's rule engine, and `verify.py` discards
any narrative that cites a passage it wasn't given, asserts a different class, or
softens the triage. Full write-up: [PART2.md](../PART2.md).